In [1]:
import tensorflow as tf
import numpy as np

from keras.src.metrics.accuracy_metrics import accuracy

from src.model_loader import ModelLoader
from src.data_loader import DataLoader
from src.utils import get_class_weigths, get_confusion_matrix, get_classification_report

print(tf.__version__)

2.16.2


In [2]:
# data_loader = DataLoader()
#
# train_b_nocw, val_b_nocw, test_b_nocw = data_loader.load_binary_dataset(positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"])
#
# train_b_cw, val_b_cw, test_b_cw = data_loader.load_binary_dataset(positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"], class_weights=True)
#
# train_mc_nocw, val_mc_nocw, test_mc_nocw = data_loader.load_multiclass_dataset(class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"])
#
# train_mc_cw, val_mc_cw, test_mc_cw = data_loader.load_multiclass_dataset(class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"], class_weights=True)

2025-04-11 08:35:14.408565: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-04-11 08:35:14.408586: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-04-11 08:35:14.408591: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-04-11 08:35:14.408604: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-11 08:35:14.408612: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-04-11 08:35:41.110814: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


Found bad path ./../datasets/Painting/painting_02662.jpg...{{function_node __wrapped__DecodeImage_device_/job:localhost/replica:0/task:0/device:CPU:0}} Input size should match (header_size + row_size * abs_height) but they differ by 2 [Op:DecodeImage] name: 


2025-04-11 08:36:27.718461: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INVALID_ARGUMENT: Input size should match (header_size + row_size * abs_height) but they differ by 2
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


KeyboardInterrupt: 

In [ ]:
# Préparation des datasets
data_loader = DataLoader()

datasets = {
    "binary_nocw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"]
    ),
    "binary_cw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
    "multiclass_nocw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"]
    ),
    "multiclass_cw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
}

# CNN_HARD

In [4]:
cnn_hard_loader = ModelLoader(model_name="CNN_HARD")

history_cnn_all_ds = []
all_model_cnn = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    cnn_hard_model = cnn_hard_loader.create_model_CNN_hard()
    all_model_cnn[f"{dataset_name}"] = cnn_hard_model

    if "nocw" in dataset_name:
        history_cnn = cnn_hard_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_model.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
        )
        history_cnn_all_ds.append(history_cnn)
    else:
        history_cnn = cnn_hard_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_model.get_early_stopping(), cnn_hard_model.get_model_checkpoint()],
        )
        history_cnn_all_ds.append(history_cnn)

# RES_NET

In [ ]:
res_net_loader = ModelLoader(model_name="RES_NET")

history_resnet_all_ds = []
all_model_resnet = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    res_net_model = res_net_loader.create_model_CNN_hard()
    all_model_resnet[f"{dataset_name}"] = res_net_model

    if "nocw" in dataset_name:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds.append(history_resnet)
    else:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds.append(history_resnet)

# INCEPTION

In [ ]:

inception_loader = ModelLoader(model_name="INCEPTION")

history_inception_all_ds = []
all_model_inception = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    inception_model = inception_loader.create_model_with_inception()
    all_model_inception[f"{dataset_name}"] = inception_model

    if "nocw" in dataset_name:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[inception_model.get_tensorboard_callback(), inception_model.get_early_stopping(), inception_model.get_model_checkpoint()],
        )
        history_inception_all_ds.append(history_inception)
    else:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[inception_model.get_tensorboard_callback(), inception_model.get_early_stopping(), inception_model.get_model_checkpoint()],
        )
        history_inception_all_ds.append(history_inception)